# Update 07 — Paired data–Mathar difference intervals vs block length

In [1]:
import pandas as pd
import numpy as np
import sys, os, time
sys.path.append('..')
from models.mathar.Mathar2007 import n as n_mathar_scalar

In [2]:
# Load data and build the Mathar cloud on identical rows

df = pd.read_csv('../../data/processed/full_data.csv')
df['time'] = pd.to_datetime(df['time'], utc=True)   # tz-aware UTC (same as nb 06)
df = df.sort_values('time').reset_index(drop=True)
N = len(df)
T_C  = df['temperature'].values        # deg C
H_pct= df['humidity'].values           # %RH
P_hPa= df['pressure'].values           # hPa
n_data = df['n_1762'].values
t_sec = df['time'].values.astype('int64') / 1e9     # epoch s (UTC, same as nb 06)

lam_um = 1.762
T_K  = T_C + 273.15
P_Pa = P_hPa * 100.0
t0 = time.time()
n_mathar = np.array([n_mathar_scalar(lam_um, Tk, pp, hh)
                     for Tk, pp, hh in zip(T_K, P_Pa, H_pct)])
print(f'N = {N};  Mathar cloud in {time.time()-t0:.1f} s')

N = 145784;  Mathar cloud in 1.5 s


In [3]:
# Segment boundaries (gap > 2 h)

GAP = 2 * 3600.0
bounds = []
seg_start = 0
for i in range(1, N):
    if t_sec[i] - t_sec[i-1] > GAP:
        bounds.append((seg_start, i-1))
        seg_start = i
bounds.append((seg_start, N-1))
print(f'segments: {len(bounds)}')
print('segment (start,end) lengths:', [(b-a+1) for (a,b) in bounds][:12], '...')

segments: 104
segment (start,end) lengths: [17, 84, 114, 175, 49, 6005, 2, 5226, 430, 2, 1, 1] ...


In [4]:
# Physical-time block builder (blocks never cross a segment gap)

def build_blocks(D, seg_bounds, t, min_frac=0.5):
    blocks = []
    dropped = 0
    for (a, b) in seg_bounds:
        i0 = a
        while i0 <= b:
            i1 = i0
            while i1 < b and t[i1] - t[i0] < D:
                i1 += 1
            span = t[i1] - t[i0]
            if span >= min_frac * D:
                blocks.append((i0, i1))
            else:
                dropped += (i1 - i0 + 1)
            i0 = i1 + 1
    return blocks, dropped

In [5]:
# Paired block bootstrap of Delta = beta_data - beta_mathar Identical resampled rows for both fits (full campaign and in-domain).

def paired_phys_boot(blocks, y_data, y_mathar, T_use, H_use, P_use,
                     B=1000, seed=42):
    rng = np.random.default_rng(seed)
    nb = len(blocks)
    XtXb = np.zeros((nb, 4, 4)); Ud = np.zeros((nb, 4)); Um = np.zeros((nb, 4))
    for bi, (a, b) in enumerate(blocks):
        Xb = np.column_stack([np.ones(b-a+1), T_use[a:b+1], H_use[a:b+1], P_use[a:b+1]])
        XtXb[bi] = Xb.T @ Xb
        Ud[bi] = Xb.T @ y_data[a:b+1]
        Um[bi] = Xb.T @ y_mathar[a:b+1]
    XtXf = XtXb.reshape(nb, 16)
    diffs = np.empty((B, 4))
    for r in range(B):
        picks = rng.integers(0, nb, size=nb)
        XtX = XtXf[picks].sum(axis=0).reshape(4, 4)
        bd = np.linalg.solve(XtX, Ud[picks].sum(axis=0))
        bm = np.linalg.solve(XtX, Um[picks].sum(axis=0))
        diffs[r] = bd - bm
    return diffs

# masks
mask_in = (T_C >= 10.0) & (T_C <= 25.0)
print(f'in-domain N = {mask_in.sum()} ({100*mask_in.sum()/N:.1f} %)')

in-domain N = 12134 (8.3 %)


In [6]:
# Block-length sweep for the PAIRED differences (full campaign)

D_choices = [2012.0, 25800.0, 86400.0, 172800.0]   # ~0.6 h, 7.2 h, 24 h, 48 h
labels = {2012.0:'0.6 h', 25800.0:'7.2 h', 86400.0:'24 h', 172800.0:'48 h'}
B = 1000

print('=== Full campaign: Delta_alpha = alpha_data - alpha_Mathar ===')
print(f"{'D':>8} {'nblk':>6} {'n_used':>8} {'frac':>6} |"
      f" {'DaT point':>11} {'DaT 95% CI':>22} |"
      f" {'DaH point':>11} {'DaH 95% CI':>22}")
for D in D_choices:
    blocks, dropped = build_blocks(D, bounds, t_sec)
    n_used = N - dropped
    d = paired_phys_boot(blocks, n_data, n_mathar, T_C, H_pct, P_hPa, B=B)
    # d columns: [dn0, daT, daH, daP]
    for j, lab in zip([1,2], ['aT','aH']):
        lo, hi = np.percentile(d[:, j], [2.5, 97.5])
        flag = '' if (lo < 0 < hi) else '  <-- excl 0'
        print(f'{D:8.0f} {len(blocks):6d} {n_used:8d} {n_used/N:6.3f} |'
              f' {d[:,j].mean():+11.3e} [{lo:+10.3e}, {hi:+10.3e}]{flag}')
    print()

=== Full campaign: Delta_alpha = alpha_data - alpha_Mathar ===
       D   nblk   n_used   frac |   DaT point             DaT 95% CI |   DaH point             DaH 95% CI
    2012   1380   145243  0.996 |  -3.128e-09 [-5.547e-09, -6.115e-10]  <-- excl 0
    2012   1380   145243  0.996 |  +4.276e-09 [+1.986e-09, +6.424e-09]  <-- excl 0

   25800    122   136468  0.936 |  -2.994e-09 [-1.115e-08, +4.452e-09]
   25800    122   136468  0.936 |  +4.993e-09 [-2.295e-09, +1.218e-08]

   86400     31    84570  0.580 |  -4.649e-09 [-2.096e-08, +6.409e-09]
   86400     31    84570  0.580 |  +4.446e-09 [-5.638e-09, +1.546e-08]

  172800      5    21881  0.150 |  +1.274e-08 [-1.640e-08, +4.730e-08]
  172800      5    21881  0.150 |  +2.754e-09 [-9.951e-09, +3.450e-08]



In [7]:
# In-domain subset (blocks restricted to in-domain rows, contiguous within)

mask_in_idx = np.where(mask_in)[0]
# Sub-segments: contiguous in-domain runs, split ALSO at time gaps > GAP so
# that physical-time blocks never straddle a data gap (identical rule to the
# full-campaign pool above). The in-domain rows form one row-contiguous run
# but contain 28 gaps > 2 h (largest ~145 h); without this split, long blocks
# would bridge those gaps (the exact defect fixed in nb 06 for full campaign).
in_bounds = []
s0 = mask_in_idx[0]; prev = mask_in_idx[0]
for idx in mask_in_idx[1:]:
    if (idx - prev > 1) or (t_sec[idx] - t_sec[prev] > GAP):
        in_bounds.append((s0, prev)); s0 = idx
    prev = idx
in_bounds.append((s0, prev))
print(f'in-domain time/row runs: {len(in_bounds)};  N = '
      f'{sum(b-a+1 for a,b in in_bounds)}')

# in-domain local time array (use original epoch seconds)
print()
print('=== In-domain: Delta_alpha ===')
for D in D_choices:
    blocks, dropped = build_blocks(D, in_bounds, t_sec)
    n_used = sum(b-a+1 for a,b in blocks)
    d = paired_phys_boot(blocks, n_data, n_mathar, T_C, H_pct, P_hPa, B=B)
    for j, lab in zip([1,2], ['aT','aH']):
        lo, hi = np.percentile(d[:, j], [2.5, 97.5])
        flag = '' if (lo < 0 < hi) else '  <-- excl 0'
        print(f'{D:8.0f} {len(blocks):6d} {n_used:8d} '
              f'{n_used/mask_in.sum():6.3f} |'
              f' {d[:,j].mean():+11.3e} [{lo:+10.3e}, {hi:+10.3e}]{flag}')
    print()


in-domain time/row runs: 29;  N = 12134

=== In-domain: Delta_alpha ===
    2012    271    12111  0.998 |  -1.444e-08 [-4.075e-08, +1.168e-08]
    2012    271    12111  0.998 |  +4.936e-09 [-3.634e-09, +1.450e-08]

   25800     26    12034  0.992 |  -1.347e-08 [-7.976e-08, +5.268e-08]
   25800     26    12034  0.992 |  +5.543e-09 [-1.308e-08, +2.965e-08]

   86400      8    11498  0.948 |  -4.895e-09 [-1.215e-07, +9.649e-08]
   86400      8    11498  0.948 |  +8.068e-09 [-2.100e-08, +3.517e-08]

  172800      3    11154  0.919 |  +8.040e-11 [-9.573e-08, +6.851e-08]
  172800      3    11154  0.919 |  +1.675e-08 [-2.255e-08, +4.709e-08]



In [8]:
# Consolidation (frozen snapshot): deterministic re-run with the fixed
#    in-domain split (seed 42 -> identical draws to the sweeps above).
#    Writes paired_diff_block_sensitivity.json next to campaign_statistics.json
#    and prints the compact table used in the block-length discussion.
import json

def sweep(name, segs, total):
    recs = []
    for D in D_choices:
        blocks, dropped = build_blocks(D, segs, t_sec)
        n_used = total - dropped
        d = paired_phys_boot(blocks, n_data, n_mathar, T_C, H_pct, P_hPa, B=B)
        r = {'D_s': D, 'D_h': round(D/3600, 2), 'n_blocks': len(blocks),
             'n_used': int(n_used), 'frac': float(n_used/total)}
        for j, lab in zip([1, 2, 3], ['aT', 'aH', 'aP']):
            lo, hi = np.percentile(d[:, j], [2.5, 97.5])
            r[lab] = {'point': float(d[:, j].mean()),
                      'ci95_lo': float(lo), 'ci95_hi': float(hi),
                      'excludes_0': bool(not (lo < 0 < hi))}
        recs.append(r)
    return recs

results = {'full_campaign': sweep('full', bounds, N),
           'in_domain': sweep('in_domain', in_bounds, int(mask_in.sum()))}

with open('paired_diff_block_sensitivity.json', 'w') as f:
    json.dump(results, f, indent=1)
print('saved paired_diff_block_sensitivity.json')

print(f"{'subset':<13}{'D':>7}{'nblk':>6}{'frac':>7} |"
      f" {'DaT point [95% CI]':>30} {'DaH point [95% CI]':>30} {'excl0 T H':>10}")
for name, recs in results.items():
    for r in recs:
        cT, cH = r['aT'], r['aH']
        print(f"{name:<13}{r['D_h']:>5.2f}h{r['n_blocks']:>6}{r['frac']:>7.3f} |"
              f" {cT['point']:+.2e} [{cT['ci95_lo']:+.2e}, {cT['ci95_hi']:+.2e}]"
              f" {cH['point']:+.2e} [{cH['ci95_lo']:+.2e}, {cH['ci95_hi']:+.2e}]"
              f" {str(cT['excludes_0']):>5} {str(cH['excludes_0']):>5}")


saved paired_diff_block_sensitivity.json
subset             D  nblk   frac |             DaT point [95% CI]             DaH point [95% CI]  excl0 T H
full_campaign 0.56h  1380  0.996 | -3.13e-09 [-5.55e-09, -6.11e-10] +4.28e-09 [+1.99e-09, +6.42e-09]  True  True
full_campaign 7.17h   122  0.936 | -2.99e-09 [-1.11e-08, +4.45e-09] +4.99e-09 [-2.29e-09, +1.22e-08] False False
full_campaign24.00h    31  0.580 | -4.65e-09 [-2.10e-08, +6.41e-09] +4.45e-09 [-5.64e-09, +1.55e-08] False False
full_campaign48.00h     5  0.150 | +1.27e-08 [-1.64e-08, +4.73e-08] +2.75e-09 [-9.95e-09, +3.45e-08] False False
in_domain     0.56h   271  0.998 | -1.44e-08 [-4.07e-08, +1.17e-08] +4.94e-09 [-3.63e-09, +1.45e-08] False False
in_domain     7.17h    26  0.992 | -1.35e-08 [-7.98e-08, +5.27e-08] +5.54e-09 [-1.31e-08, +2.97e-08] False False
in_domain    24.00h     8  0.948 | -4.89e-09 [-1.21e-07, +9.65e-08] +8.07e-09 [-2.10e-08, +3.52e-08] False False
in_domain    48.00h     3  0.919 | +8.04e-11 [-9.57e-08, +6

## Summary (2026-09-08) — paired Δα intervals vs block length

**Implementation note** The in-domain rows form a single row-contiguous run that nevertheless contains **28 time gaps > 2 h** (largest ≈ 145 h). Blocks were therefore built across data gaps there, unlike the full-campaign pool. In-domain sub-segments are now split at time gaps as well (rule parity); block counts dropped (24 h: 19 → 8; 48 h: 14 → 3) because short run tails are no longer bridged across gaps. Conclusions are unchanged.

| subset | D | n_blocks | frac | Δα_T point [95 % CI] ×10⁻⁹ | Δα_H point [95 % CI] ×10⁻⁹ | excl. 0? |
|---|---|---|---|---|---|---|
| full | 0.56 h | 1,380 | 0.996 | −3.13 [−5.55, −0.61] | +4.28 [+1.99, +6.42] | T ✓ H ✓ |
| full | 7.2 h | 122 | 0.936 | −2.99 [−11.1, +4.45] | +4.99 [−2.29, +12.2] | — |
| full | 24 h | 31 | 0.580 | −4.65 [−21.0, +6.41] | +4.45 [−5.64, +15.5] | — |
| full | 48 h | 5 | 0.150 | +12.7 [−16.4, +47.3] | +2.75 [−9.95, +34.5] | — |
| in-domain | 0.56 h | 271 | 0.998 | −14.4 [−40.7, +11.7] | +4.94 [−3.63, +14.5] | — |
| in-domain | 7.2 h | 26 | 0.992 | −13.5 [−79.8, +52.7] | +5.54 [−13.1, +29.7] | — |
| in-domain | 24 h | 8 | 0.948 | −4.89 [−121, +96.5] | +8.07 [−21.0, +35.2] | — |
| in-domain | 48 h | 3 | 0.919 | +0.08 [−95.7, +68.5] | +16.7 [−22.5, +47.1] | — |

**Findings.**
1. Full campaign: the *point* estimates are stable across block lengths (Δα_H ≈ +4.3…+5.0 ×10⁻⁹; Δα_T ≈ −3.0…−4.7 ×10⁻⁹); what changes is the uncertainty. Exclusion of zero occurs **only** at 0.56 h blocks (≈ the legacy L = 156 row block; its SE ≈ 1.13 ×10⁻⁹ cross-validates the HAC L = 156 value 1.15 ×10⁻⁹).
2. At 7.2 h and 24 h both Δα_T and Δα_H intervals contain zero.
3. 48 h is unusable for the full campaign (5 blocks, 15 % coverage; the Δα_T point even flips sign) — retained only to show the degeneracy.
4. In-domain: no block length yields an interval excluding zero (N = 12,134, power limited), consistent with earlier in-domain results.